# Making a `sandsuet`-compliant dataset with multiple groups

NetCDF4 allows for the concept of hierarchical groups that function as top-level directories within any dataset. There are many times when one wishes to separate variables out into separate groups, and maybe even nest them hierarchically. This is implemented in the NetCDF standard as a kind of filetree within a file. Thus, the top level group is always the root, and groups nested below that follow a path-based addressing system. Each group functions as an independent dataset, but groups lower in the hierarchy can inherit variables/dimensions/attributes from groups higher in the hierarchy. 

For a dataset to be `sandsuet` compliant, the data and variables at the top level of the file must be have the largest rank (most dimensions) and be the main product the creator intends to share. (section §1.a). This means that your main data should be stored at the root group and auxiliary data need to go further down the project tree.

This tutorial shows you how to assemble a `sandsuet`-compliant dataset with groups using the python `xarray` module. 

## Making a fake dataset

First, we will borrow from the fake data example to generate a small fake dataset, but we will divide the time-series data into a separate group. First, let us import the needed packages and create dummy variables for everything.

In [ ]:
import numpy as np
import xarray as xr

import os

import matplotlib.pyplot as plt

In [ ]:
# some spatial information
x = np.arange(-6, 6, 0.1)
y = np.arange(-3, 3, 0.1)

# some temporal information
t = np.arange(1, 5)

# meshes for each to make fake data
T, Y, X = np.meshgrid(t, y, x, indexing="ij")

# make fake time by x by y data
#   hint: this would be the data from a model, experiment, or the field
eta = np.sin(T * X + Y)
velocity = np.cos(T * Y + X)

# make fake sea-level curve
H_SL = np.linspace(0.25, 0.9, num=len(t))  # sea level

# make fake concentration time-series
conc = np.geomspace(0.1, 25, num=len(t)) # an exponentially increasing concentration 

Now, `xarray` achieves the notion of groups using a structure called a DataTree. Each node is its own xarray DataSet. I will create two data arrays, one for the top level data, and the other for the auxiliary data. They will share a coordinate description dictionary and encoding dictionary. 

The encoding dictionary is not strictly necessary, but this shows you how to strictly adhere to § 6.f of the `sandsuet` spec and ensure that the missing values are consistent across all data objects. 

In [ ]:
top_level_data = {
    'eta':(
        ['t', 'y', 'x'], eta, 
        {
            'units':'meter', 
            'long_name':'channel_bottom__elevation'
        },
        dict(_FillValue=np.nan, dtype='f4')

    ),
    'velocity':(
        ['t', 'y', 'x'], 
        velocity, 
        {
            'units':'meter/second', 
            'long_name':'channel_water_flowing__speed'
        },
        dict(_FillValue=np.nan, dtype='f4')

    )
}

In [ ]:
aux_data = {
    'sea_level':(
        ['t'], 
        H_SL, 
        {
            'units':'meter', 
            'long_name':'basin_water_surface__elevation'
        },
        dict(_FillValue=np.nan, dtype='f4')

    ),
    'concentration':(
        ['t'], 
        conc, 
        {
            'units':'mol/L',
            'long_name':'carbon_dioxide__concentration'
        },
        dict(_FillValue=np.nan, dtype='f4')

    )
}

In [ ]:
coordinates = {
    't':(
        ['t'], 
        t,
        {
            'units':'seconds',
            'long_name':'Time_dimension'
        },
        dict(_FillValue=np.nan, dtype='f4')

    ),
    'x':(
        ['x'],
        x,
        {
            'units':'meters',
            'long_name':'X_dimension'
        },
        dict(_FillValue=np.nan, dtype='f4')

    ),
    'y':(
        ['y'],
        y,
        {
            'units':'meters',
            'long_name':'Y_dimension'
        },
        dict(_FillValue=np.nan, dtype='f4')
    )
}

global_attributes = dict(
    description = "Output from MyFakeModel",
    source = "MyFakeModel v0.1",
    sandsuet_version = "1.0.0",
)

Now, let's combine the components into our two DataSets

In [ ]:
top_level_dataset = xr.Dataset(
    data_vars=top_level_data,
    coords=coordinates,
    attrs=global_attributes
)

aux_dataset = xr.Dataset(
    data_vars=aux_data,
    coords=coordinates,
    # attrs=global_attributes
)

And now to combine them in a hierarchical tree. The names are important, and there are two ways to do it. I will first show the most clear method, where you start with a dictionary.

In [ ]:
data_tree_dict = xr.DataTree.from_dict(
    {
        "/":top_level_dataset,
        "/aux":aux_dataset
    }
)

# view contents
print(data_tree_dict)

The second way is more cumbersome and involves more calls, but can be done. The key is that the child node has to be made before the parent. 

In [ ]:
data_tree_two = xr.DataTree(aux_dataset)
data_tree_one = xr.DataTree(top_level_dataset, children={'aux':data_tree_two})

print(data_tree_one)


Either way, now the file can be saved as a `sandsuet`-compliant NetCDF

In [ ]:
directory = os.getcwd()
file_path = os.path.join(directory, "fakemodel_output_xarray_groups.nc")


In [ ]:
data_tree_dict.to_netcdf(file_path)